# 01 · Ingestão e casamento das fontes — IBGE × ANEEL × INPE

**Objetivo:** baixar as três fontes públicas e montar uma única tabela municipal que cruza
**recurso** (irradiação GTI do INPE) com **uso real** (potência fotovoltaica instalada da ANEEL),
sobre os polígonos do **IBGE**.

Este é o núcleo do projeto. O desafio não é plotar — é **casar bases com chaves geográficas
diferentes**:

| Fonte | Granularidade | Chave |
|---|---|---|
| INPE/LABREN | grade de ~10 km (lon/lat) | nenhum código de município |
| ANEEL | empreendimento | `CodMunicipioIbge` (código IBGE) |
| IBGE (geobr) | polígono municipal | `code_muni` |

**Estratégia:** INPE→IBGE por **geometria** (ponto-em-polígono); ANEEL→IBGE por **código**
(nunca por nome — acento/grafia divergem).

> ⚠️ **Licença:** os dados do INPE/LABREN não podem ser redistribuídos. Nada baixado aqui é
> versionado (ver `.gitignore`). Cite: *LABREN/CCST/INPE — Brasil, Atlas Brasileiro de Energia Solar, 2ª ed., 2017.*

## 0 · Setup

Importa os módulos de `src/` e habilita `autoreload` para editar o código sem reiniciar o kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Permite importar o pacote src/ a partir do notebook (que roda em notebooks/).
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import geopandas as gpd

from src import config, download, ingest_inpe, ingest_aneel, ingest_ibge, aggregate

print("Raiz do projeto:", ROOT)
print("Dados em:", config.DATA)

## 1 · INPE/LABREN — grade de irradiação (GTI)

Baixamos a variável **TILTED (plano inclinado / GTI)** — a que o painel realmente capta.
O arquivo é um ZIP com um CSV (`;` separador, `.` decimal), médias em **Wh/m².dia**.

Na **primeira execução**, conferimos o cabeçalho real antes de confiar nos nomes de coluna.

In [ ]:
# Baixa e descompacta o GTI (cacheado em data/).
arquivos_inpe = download.baixar_inpe("gti")

# Confere o cabeçalho real do CSV (nomes de coluna e tipos).
csv_inpe = next(p for p in arquivos_inpe if str(p).lower().endswith(".csv"))
ingest_inpe.inspecionar_cabecalho(csv_inpe)

In [ ]:
# Carrega a grade como GeoDataFrame de pontos (EPSG:4326). Converte Wh -> kWh/m².dia.
pontos_gti = ingest_inpe.carregar_grade("gti")
pontos_gti.head()

## 2 · ANEEL — potência fotovoltaica instalada por município

O CSV da ANEEL é grande (ZIP ~122 MB). Lemos em **chunks**, filtramos só fotovoltaica
(`SigTipoGeracao == 'UFV'`) e somamos `MdaPotenciaInstaladakW` por `CodMunicipioIbge`.

> O CSV usa `;` e **vírgula decimal**, encoding `latin-1` (padrão gov.br). Tudo isso já está
> tratado em `ingest_aneel`.

In [ ]:
download.baixar_aneel()
aneel_mun = ingest_aneel.agregar_por_municipio()
aneel_mun.sort_values("pot_instalada_mw", ascending=False).head(10)

## 3 · IBGE — polígonos municipais e população

Via `geobr`. O GeoDataFrame já vem com `code_muni` (chave!) e `name_muni`.
A população (API de agregados) é necessária para o indicador **per capita**.

In [ ]:
municipios = ingest_ibge.carregar_municipios()
estados = ingest_ibge.carregar_estados()
populacao = ingest_ibge.carregar_populacao()
municipios.head(3)

## 4 · Junção espacial INPE → IBGE (ponto-em-polígono)

Cada ponto da grade recebe o município que o contém → **média de GTI por município**.
Municípios pequenos sem ponto interno (a grade é de 10 km) entram pelo **fallback de
centroide** (ponto da grade mais próximo), garantindo cobertura total.

In [ ]:
# 4a) Média por polígono.
gti_mun = aggregate.media_gti_por_municipio(pontos_gti, municipios)

# 4b) Fallback para municípios sem nenhum ponto interno.
faltantes = municipios.loc[~municipios["code_muni"].isin(gti_mun["code_muni"]), "code_muni"]
print(f"Municípios sem ponto interno (vão para fallback): {faltantes.nunique():,}")
gti_fb = aggregate.gti_por_centroide(pontos_gti, municipios, faltantes)

gti_mun_full = pd.concat([gti_mun, gti_fb], ignore_index=True)
print(f"GTI definido para {gti_mun_full['code_muni'].nunique():,} municípios.")

## 5 · Cruzamento recurso × uso + índice de oportunidade

Une GTI (recurso) + potência FV (uso) + população na malha municipal. Municípios sem
instalação recebem potência **0** (ausência real). O `score_oportunidade` = percentil de
recurso − percentil de uso destaca os **desertos de aproveitamento** (muito sol, pouco uso).

In [ ]:
atlas = aggregate.cruzar(gti_mun_full, aneel_mun, populacao, municipios)
atlas = aggregate.indice_aproveitamento(atlas)

print(atlas.shape)
atlas[["code_muni", "name_muni", "abbrev_state", "gti_anual",
       "pot_instalada_mw", "w_per_capita", "score_oportunidade",
       "classe_oportunidade"]].head()

## 6 · Sanidade + persistência

Checagens rápidas antes de salvar a tabela final em `data/processed` (GeoParquet).

In [ ]:
# Sanidade: cobertura de GTI, total de potência, e prévia dos extremos.
print("Municípios sem GTI:", int(atlas["gti_anual"].isna().sum()))
print("Municípios com FV  :", int((atlas["pot_instalada_mw"] > 0).sum()))
print("Potência FV total  : %.0f MW" % atlas["pot_instalada_mw"].sum())

print("\nTop 5 GTI (maior recurso):")
display(atlas.nlargest(5, "gti_anual")[["name_muni", "abbrev_state", "gti_anual"]])

print("Top 5 'desertos de aproveitamento' (alto recurso, baixo uso):")
desertos = atlas[atlas["classe_oportunidade"] == "deserto de aproveitamento"]
display(desertos.nlargest(5, "score_oportunidade")[
    ["name_muni", "abbrev_state", "gti_anual", "w_per_capita", "score_oportunidade"]])

In [ ]:
# Salva a tabela final (GeoParquet preserva geometria + tipos).
saida = config.PROCESSED / "atlas_municipios.parquet"
atlas.to_parquet(saida)
print("Salvo em:", saida)

## Próximos passos

- **02 · EDA + mapas de recurso** — coroplético de GTI, sazonalidade mensal por região, ranking.
- **03 · Cruzamento (insight)** — mapa dos desertos de aproveitamento, GTI × W/hab.
- **README** — escrever os 3–4 achados com os mapas gerados.

A tabela `data/processed/atlas_municipios.parquet` é a base para os próximos notebooks.